# T65-自动总结文档内容

本Notebook实现文档自动总结功能，包括：
- 文档OCR识别（支持PDF、DOCX、EPUB等格式）
- 使用通义千问Qwen-Long进行内容总结
- 敏感词过滤
- 邮件通知
- 数据库存储

## 功能说明

1. **文档处理**：支持多种文档格式的OCR识别
2. **AI总结**：使用阿里云Qwen-Long模型进行智能总结
3. **敏感词过滤**：使用Aho-Corasick算法过滤敏感词
4. **邮件通知**：自动发送总结结果到指定邮箱

## 1. 环境配置与依赖导入

导入所需的库并加载配置文件。

In [ ]:
# 环境配置与依赖导入
import os
import sys
import re
import io
import json
import base64
import shutil
import logging
import tempfile
import concurrent.futures
from pathlib import Path
from datetime import datetime

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text

# 图像处理
from PIL import Image
import cv2

# 文档处理
import fitz  # PyMuPDF
from docx import Document
from ebooklib import epub
from tika import parser

# OCR
try:
    from paddleocr import PaddleOCR, draw_ocr
    PADDLE_AVAILABLE = True
except ImportError:
    PADDLE_AVAILABLE = False
    import pytesseract
    
# 网页解析
from bs4 import BeautifulSoup

# OpenAI客户端（用于Qwen-Long）
import openai

# 邮件
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# 敏感词过滤
import ahocorasick

# Selenium
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service as ChromeService
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

# 导入配置
from config import (
    get_db_connection_string,
    get_info_db_connection_string,
    QWEN_API_KEY,
    QWEN_BASE_URL,
    SMTP_SERVER,
    SMTP_PORT,
    EMAIL_USER,
    EMAIL_PASSWORD,
    RECIPIENT_EMAIL,
    INPUT_FOLDERS,
    OUTPUT_FOLDER,
    SENSITIVE_WORDS_FILE
)

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 设置环境变量
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

print("依赖导入完成")

## 2. 数据库与API配置

配置数据库连接和Qwen API客户端。

In [ ]:
# 数据库与API配置

# 创建数据库引擎
main_db_engine = sqlalchemy.create_engine(
    get_db_connection_string(),
    poolclass=sqlalchemy.pool.NullPool,
    pool_recycle=3600
)

info_db_engine = sqlalchemy.create_engine(
    get_info_db_connection_string(),
    poolclass=sqlalchemy.pool.NullPool,
    pool_recycle=3600
)

print("数据库连接创建成功")

# 创建Qwen API客户端
qwen_client = openai.OpenAI(
    api_key=QWEN_API_KEY,
    base_url=QWEN_BASE_URL
)
print("Qwen API客户端创建成功")

## 3. 敏感词过滤

实现敏感词加载和过滤功能。

In [ ]:
# 敏感词过滤

def read_sensitive_words(filename):
    """读取敏感词列表
    
    Args:
        filename: 敏感词文件路径
        
    Returns:
        list: 敏感词列表
    """
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            sensitive_words = [line.strip() for line in file]
        return sensitive_words
    except FileNotFoundError:
        logging.warning(f"敏感词文件未找到: {filename}")
        return []

def build_automaton(sensitive_words):
    """构建Aho-Corasick自动机
    
    Args:
        sensitive_words: 敏感词列表
        
    Returns:
        Automaton: Aho-Corasick自动机
    """
    A = ahocorasick.Automaton()
    for idx, word in enumerate(sensitive_words):
        A.add_word(word, (idx, word))
    A.make_automaton()
    return A

def filter_sensitive_words(text, automaton):
    """过滤文本中的敏感词
    
    Args:
        text: 待过滤文本
        automaton: Aho-Corasick自动机
        
    Returns:
        str: 过滤后的文本
    """
    matches = []
    for end_index, (insert_order, original_value) in automaton.iter(text):
        start_index = end_index - len(original_value) + 1
        matches.append((start_index, end_index + 1))

    filtered_text_parts = []
    last_end = 0
    for start, end in matches:
        filtered_text_parts.append(text[last_end:start])
        last_end = end
    filtered_text_parts.append(text[last_end:])

    return ''.join(filtered_text_parts)

# 加载敏感词
sensitive_words_list = read_sensitive_words(SENSITIVE_WORDS_FILE)
automaton = build_automaton(sensitive_words_list)
print(f"敏感词加载完成，共 {len(sensitive_words_list)} 个")

## 4. OCR识别功能

实现文档OCR识别功能，支持多种文档格式。

In [ ]:
# OCR识别功能

# 初始化OCR引擎
if PADDLE_AVAILABLE:
    try:
        ocr = PaddleOCR(
            use_angle_clsf=True,
            lang='ch',
            show_log=False,
            use_gpu=True
        )
        OCR_METHOD = 'paddle'
        print("PaddleOCR初始化成功")
    except Exception as e:
        logging.warning(f"PaddleOCR初始化失败: {e}")
        OCR_METHOD = 'tesseract'
else:
    OCR_METHOD = 'tesseract'
    print("使用Tesseract OCR")

class DocumentProcessor:
    """文档处理器"""
    
    @staticmethod
    def recognize_and_draw(img):
        """使用OCR识别图像中的文字
        
        Args:
            img: 图像数组
            
        Returns:
            str: 识别的文本
        """
        try:
            if OCR_METHOD == 'paddle':
                results = ocr.ocr(img, cls=True)
                result = results[0] if results else []
                text = ''
                for idx in range(len(result)):
                    try:
                        res = result[idx]
                        text += res[1][0] + '\n'
                    except Exception:
                        pass
                return text
            else:
                return pytesseract.image_to_string(
                    img,
                    config='--oem 3 --psm 6 -l chi_sim+eng'
                )
        except Exception as e:
            logging.error(f"OCR识别失败: {e}")
            return ''
            
    @staticmethod
    def preprocess_and_ocr_image(img):
        """图像预处理并OCR
        
        Args:
            img: 图像数组
            
        Returns:
            str: 识别的文本
        """
        try:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(
                gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU
            )
            _, dist = cv2.threshold(
                thresh, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU
            )
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 1))
            opening = cv2.morphologyEx(dist, cv2.MORPH_OPEN, kernel)
            return DocumentProcessor.recognize_and_draw(opening)
        except Exception as e:
            logging.error(f"图像预处理失败: {e}")
            return ''
            
    @staticmethod
    def get_pdf_text_and_images(pdf_path):
        """提取PDF文本和图像
        
        Args:
            pdf_path: PDF文件路径
            
        Returns:
            tuple: (文本, 图像文本列表)
        """
        try:
            # 使用Tika提取文本
            parsed = parser.from_file(pdf_path)
            text_extracted = parsed.get('content', '')
            
            # 提取图像
            doc = fitz.open(pdf_path)
            images_list = []
            for page in doc:
                for img in page.get_images(full=True):
                    xref = img[0]
                    base_image = doc.extract_image(xref)
                    images_list.append(base_image)
                    
            # OCR图像
            image_texts = []
            for idx, image in enumerate(images_list, start=1):
                try:
                    image_data = image["image"]
                    img = Image.open(io.BytesIO(image_data))
                    img_array = np.array(img)
                    ocr_result = DocumentProcessor.preprocess_and_ocr_image(img_array)
                    image_texts.append((f"---IMAGE {idx}---", ocr_result))
                except Exception as e:
                    logging.error(f"图像处理失败: {e}")
                    
            return text_extracted, image_texts
            
        except Exception as e:
            logging.error(f"PDF处理失败: {e}")
            return '', []
            
    @staticmethod
    def get_docx_text_and_images(docx_path):
        """提取DOCX文本和图像
        
        Args:
            docx_path: DOCX文件路径
            
        Returns:
            tuple: (文本, 图像文本列表)
        """
        try:
            document = Document(docx_path)
            text_extracted = "\n".join([para.text for para in document.paragraphs])
            
            # 处理图像
            image_texts = []
            image_counter = 1
            for rel in document.part.rels:
                try:
                    if "image" in document.part.rels[rel].target_ref:
                        image_data = document.part.rels[rel].target_part.blob
                        image = Image.open(io.BytesIO(image_data))
                        img_array = np.array(image)
                        ocr_result = DocumentProcessor.preprocess_and_ocr_image(img_array)
                        image_texts.append((f"---IMAGE {image_counter}---", ocr_result))
                        image_counter += 1
                except Exception as e:
                    logging.error(f"图像处理失败: {e}")
                    
            return text_extracted, image_texts
            
        except Exception as e:
            logging.error(f"DOCX处理失败: {e}")
            return '', []
            
    @staticmethod
    def process_file(file_path):
        """处理单个文件
        
        Args:
            file_path: 文件路径
            
        Returns:
            str: 处理后的文本
        """
        if file_path.endswith('.pdf'):
            text, images = DocumentProcessor.get_pdf_text_and_images(file_path)
        elif file_path.endswith('.docx') or file_path.endswith('.doc'):
            text, images = DocumentProcessor.get_docx_text_and_images(file_path)
        elif file_path.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
            images = []
        elif file_path.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            image = Image.open(file_path)
            text = DocumentProcessor.preprocess_and_ocr_image(np.array(image))
            images = []
        else:
            raise ValueError(f"不支持的文件格式: {file_path}")
            
        # 合并文本
        final_text = text + '\n下面是文档中所有图表内容：\n'
        for idx, ocr_text in enumerate(images):
            final_text = final_text + f'\n{str(ocr_text)}\n'
            
        return final_text

print("DocumentProcessor类定义完成")

## 5. Qwen-Long 总结功能

使用阿里云Qwen-Long模型进行文档总结。

In [ ]:
# Qwen-Long 总结功能

def upload_file_to_qwen(file_path):
    """上传文件到Qwen
    
    Args:
        file_path: 文件路径
        
    Returns:
        str: 文件ID
    """
    try:
        file_object = qwen_client.files.create(
            file=Path(file_path),
            purpose="file-extract"
        )
        return file_object.id
    except Exception as e:
        logging.error(f"文件上传失败: {e}")
        return None

def summarize_content(file_id, content_type=1):
    """使用Qwen-Long总结内容
    
    Args:
        file_id: 文件ID
        content_type: 内容类型（1=普通文档，2=群聊记录）
        
    Returns:
        str: 总结内容
    """
    if content_type == 1:
        prompt_text = '''请你帮我总结一下这篇文章的主要内容，请保证结构清晰、内容准确、逻辑通顺，
        并且确保不能遗漏重要内容，同时不要纳入免责声明、广告、团队介绍、联系信息等无关内容。
        用中文回答。'''
    else:
        prompt_text = '''作为专业的金融领域分析师，你的任务是分析并总结每个群聊的主要讨论内容。
        请确保总结结构清晰、内容准确、逻辑通顺。'''
        
    try:
        completion = qwen_client.chat.completions.create(
            model="qwen-long",
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'system', 'content': f'fileid://{file_id}'},
                {'role': 'user', 'content': prompt_text}
            ],
            stream=False
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        logging.error(f"内容总结失败: {e}")
        return ''

print("Qwen总结函数定义完成")

## 6. 邮件发送功能

实现邮件通知功能。

In [ ]:
# 邮件发送功能

def send_email(subject, html_content):
    """发送邮件
    
    Args:
        subject: 邮件主题
        html_content: HTML内容
    """
    try:
        msg = MIMEMultipart('alternative')
        msg['Subject'] = subject
        msg['From'] = EMAIL_USER
        msg['To'] = RECIPIENT_EMAIL

        part = MIMEText(html_content, 'html')
        msg.attach(part)

        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
            server.connect(SMTP_SERVER, SMTP_PORT)
            server.login(EMAIL_USER, EMAIL_PASSWORD)
            server.sendmail(EMAIL_USER, RECIPIENT_EMAIL, msg.as_string())
            
        logging.info(f"邮件发送成功: {subject}")
    except Exception as e:
        logging.error(f"邮件发送失败: {e}")

print("邮件发送函数定义完成")

## 7. 主程序执行

执行完整的文档处理流程。

In [ ]:
# 主程序执行

def upload_metadata_to_mysql(file_id, summary, filename):
    """上传元数据到MySQL
    
    Args:
        file_id: 文件ID
        summary: 总结内容
        filename: 文件名
    """
    sql = """
    INSERT IGNORE INTO 信息库 (dt, fileid, summary, filename)
    VALUES (:dt, :fileid, :summary, :filename)
    """
    
    params = {
        "dt": datetime.now(),
        "fileid": file_id,
        "summary": summary,
        "filename": filename
    }
    
    with info_db_engine.begin() as connection:
        connection.execute(text(sql), params)
        
    # 发送邮件通知
    summary_html = summary.replace('\n', '<br>')
    summary_html = f'fileid://{file_id}<br><br>' + summary_html
    send_email(filename, summary_html)

def process_files_in_folder(folder_path):
    """处理文件夹中的所有文件
    
    Args:
        folder_path: 文件夹路径
    """
    if not os.path.exists(folder_path):
        logging.warning(f"文件夹不存在: {folder_path}")
        return
        
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        if not os.path.isfile(file_path):
            continue
            
        # 判断内容类型
        content_type = 2 if '微信消息' in filename else 1
        
        try:
            # 处理文档
            text = DocumentProcessor.process_file(file_path)
            
            # 过滤敏感词
            text = filter_sensitive_words(text, automaton)
            
            # 保存临时文件
            base_name = os.path.splitext(filename)[0]
            temp_file_path = f"{base_name}.txt"
            with open(temp_file_path, 'w', encoding='utf-8') as f:
                f.write(text)
                
            # 上传到Qwen
            file_id = upload_file_to_qwen(temp_file_path)
            
            if file_id:
                # 获取总结
                summary = summarize_content(file_id, content_type)
                
                # 上传到数据库
                upload_metadata_to_mysql(file_id, summary, base_name + '.txt')
                
                # 备份文件
                backup_path = os.path.join(OUTPUT_FOLDER, filename)
                if not os.path.exists(backup_path):
                    shutil.copy(file_path, backup_path)
                    
                # 删除原文件和临时文件
                os.remove(file_path)
                os.remove(temp_file_path)
                
                logging.info(f"文件处理完成: {filename}")
                
        except Exception as e:
            logging.error(f"处理文件失败 [{filename}]: {e}")

def main():
    """主执行函数"""
    print("=" * 50)
    print("T65-自动总结文档内容 任务开始")
    print(f"执行时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 50)
    
    # 处理所有输入文件夹
    for folder_path in INPUT_FOLDERS:
        print(f"\n处理文件夹: {folder_path}")
        process_files_in_folder(folder_path)
        
    print("\n" + "=" * 50)
    print("T65-自动总结文档内容 任务完成")
    print("=" * 50)

# 执行主程序
if __name__ == '__main__':
    main()

## 总结

本Notebook实现了以下功能：

1. **文档处理**：支持PDF、DOCX、TXT、图片等多种格式
2. **OCR识别**：使用PaddleOCR或Tesseract进行图像文字识别
3. **AI总结**：使用阿里云Qwen-Long模型进行智能总结
4. **敏感词过滤**：使用Aho-Corasick算法过滤敏感词
5. **邮件通知**：自动发送总结结果到指定邮箱
6. **数据库存储**：将结果存储到MySQL数据库

### 使用说明

1. 确保已安装所有依赖包：`pip install -r requirements.txt`
2. 配置环境变量或修改config.py中的默认值
3. 配置敏感词文件路径
4. 运行主程序执行完整流程